In [ ]:
# divide the mcQTSA dataset into training, validation and test sets

import json
import random
import re
import os

input_file = "mcQTSA_new2.jsonl"  
train_output = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_train.jsonl"
val_output = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_val.jsonl"
test_output = "SFT/Qwen3-0.6B/testbench/mcQTSA_test.jsonl"

# Create directories if they don't exist
os.makedirs(os.path.dirname(train_output), exist_ok=True)
os.makedirs(os.path.dirname(test_output), exist_ok=True)

# Load all data
data = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

# Shuffle data
random.shuffle(data)

print(f"Total dataset size: {len(data)} pairs")


if len(data) < 1000:
    print(f"Error: Dataset only has {len(data)} pairs, but you requested 1000 test pairs!")
    exit(1)


test_data = data[:1000]
remaining_data = data[1000:]


val_size = int(len(remaining_data) * 0.1)  
train_data = remaining_data[val_size:]    
val_data = remaining_data[:val_size]       

# Write training set (QTSA format)
with open(train_output, "w", encoding="utf-8") as f_train:
    for item in train_data:
        train_item = {
            "question": item.get("question", ""),
            "thinking": item.get("thinking", ""),
            "solution": item.get("solution", ""),
            "answer": item.get("answer", "")
        }
        f_train.write(json.dumps(train_item, ensure_ascii=False) + "\n")

# Write validation set (QTSA format)
with open(val_output, "w", encoding="utf-8") as f_val:
    for item in val_data:
        val_item = {
            "question": item.get("question", ""),
            "thinking": item.get("thinking", ""),
            "solution": item.get("solution", ""),
            "answer": item.get("answer", "")
        }
        f_val.write(json.dumps(val_item, ensure_ascii=False) + "\n")

# Extract content inside <answer></answer>
def extract_answer_content(answer_text: str):
    match = re.search(r"<answer>(.*?)</answer>", answer_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

# Write test set (QA format only)
with open(test_output, "w", encoding="utf-8") as f_test:
    for idx, item in enumerate(test_data, start=1):
        answer_text = item.get("answer", "")
        answer_content = extract_answer_content(answer_text)
        test_item = {
            "id": idx,
            "question": item.get("question", ""),
            "answer": answer_content
        }
        f_test.write(json.dumps(test_item, ensure_ascii=False) + "\n")

print(f"Train dataset: {len(train_data)} pairs → {train_output}")
print(f"Validation dataset: {len(val_data)} pairs → {val_output}")
print(f"Test dataset: {len(test_data)} pairs → {test_output}")

In [ ]:
# Assign original IDs to each question in the testbench

import json
import re

# File paths
original_file = "mcQTSA_new2.jsonl"
test_file = "SFT/Qwen3-0.6B/testbench/mcQTSA_test.jsonl"
test_change_output = "SFT/Qwen3-0.6B/testbench/mcQTSA_test_change.jsonl"

# Read original file and build a mapping from question to original ID
original_questions = {}

with open(original_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f, start=1):
        if line.strip():
            item = json.loads(line)
            question = item.get("question", "").strip()
            original_questions[question] = idx  # Store original ID

print(f"Found {len(original_questions)} questions in the original file")

# Read test set file and match original IDs while preserving order
test_data_with_original_ids = []

with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_item = json.loads(line)
            question = test_item.get("question", "").strip()

            # Look up the corresponding ID in the original dataset
            original_id = original_questions.get(question)

            if original_id is not None:
                new_item = {
                    "id": original_id,  # Use original ID
                    "question": test_item["question"],
                    "answer": test_item["answer"]
                }
                test_data_with_original_ids.append(new_item)
            else:
                print(f"Warning: Original ID not found for question: {question[:50]}...")

# Save to new file while preserving original order
with open(test_change_output, "w", encoding="utf-8") as f:
    for item in test_data_with_original_ids:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Successfully matched original IDs for {len(test_data_with_original_ids)} questions")
print(f"New test set saved to: {test_change_output}")

print("ID changes for the first 5 QA pairs:")

for i in range(min(5, len(test_data_with_original_ids))):
    print(f"  Original ID: {i+1} → New ID: {test_data_with_original_ids[i]['id']}")

In [ ]:
# Assign original IDs to each question in the mcQTSA training dataset

import json
import re

# File paths
original_file = "mcQTSA_new2.jsonl"
test_file = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_train.jsonl"
test_change_output = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_train_change.jsonl"

# Read original file and build a mapping from question to original ID
original_questions = {}

with open(original_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f, start=1):
        if line.strip():
            item = json.loads(line)
            question = item.get("question", "").strip()
            original_questions[question] = idx  # Store original ID

print(f"Found {len(original_questions)} questions in the original file")

# Read training dataset and only update the ID field
# Keep all other fields unchanged
test_data_with_original_ids = []
matched_count = 0
unmatched_count = 0

with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_item = json.loads(line)
            question = test_item.get("question", "").strip()

            # Look up the corresponding ID in the original dataset
            original_id = original_questions.get(question)

            if original_id is not None:
                # Update only the ID field and preserve everything else
                test_item["id"] = original_id
                matched_count += 1
            else:
                # Keep original ID if no match is found
                print(
                    f"Warning: Original ID not found for question: {question[:50]}..."
                )
                unmatched_count += 1

            test_data_with_original_ids.append(test_item)

# Save to a new file while preserving original order and all fields
with open(test_change_output, "w", encoding="utf-8") as f:
    for item in test_data_with_original_ids:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Matching statistics:")
print(f"  Successfully matched: {matched_count} questions")
print(f"  Unmatched: {unmatched_count} questions")
print(f"  Total: {len(test_data_with_original_ids)} questions")

print(f"Updated training dataset saved to: {test_change_output}")

# Display the first 5 matching results
print("\nFirst 5 matching results:")

for i in range(min(5, len(test_data_with_original_ids))):
    item = test_data_with_original_ids[i]

    original_question = (
        list(original_questions.keys())[
            list(original_questions.values()).index(item["id"])
        ]
        if item["id"] in original_questions.values()
        else "Unknown"
    )

    print(
        f"  New ID: {item['id']}, "
        f"Question: {item['question'][:30]}..."
    )

In [ ]:
# Assign original IDs to each question in the ndQTSA training dataset

import json
import re

# File paths
original_file = "mcQTSA_new2.jsonl"
test_file = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_val.jsonl"
test_change_output = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_val_change.jsonl"

# Read original file and build a mapping from question to original ID
original_questions = {}

with open(original_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f, start=1):
        if line.strip():
            item = json.loads(line)
            question = item.get("question", "").strip()
            original_questions[question] = idx  # Store original ID

print(f"Found {len(original_questions)} questions in the original file")

# Read training dataset and only update the ID field
# Keep all other fields unchanged
test_data_with_original_ids = []
matched_count = 0
unmatched_count = 0

with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_item = json.loads(line)
            question = test_item.get("question", "").strip()

            # Look up the corresponding ID in the original dataset
            original_id = original_questions.get(question)

            if original_id is not None:
                # Update only the ID field and preserve everything else
                test_item["id"] = original_id
                matched_count += 1
            else:
                # Keep original ID if no match is found
                print(
                    f"Warning: Original ID not found for question: {question[:50]}..."
                )
                unmatched_count += 1

            test_data_with_original_ids.append(test_item)

# Save to a new file while preserving original order and all fields
with open(test_change_output, "w", encoding="utf-8") as f:
    for item in test_data_with_original_ids:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Matching statistics:")
print(f"  Successfully matched: {matched_count} questions")
print(f"  Unmatched: {unmatched_count} questions")
print(f"  Total: {len(test_data_with_original_ids)} questions")

print(f"Updated training dataset saved to: {test_change_output}")

# Display the first 5 matching results
print("\nFirst 5 matching results:")

for i in range(min(5, len(test_data_with_original_ids))):
    item = test_data_with_original_ids[i]

    original_question = (
        list(original_questions.keys())[
            list(original_questions.values()).index(item["id"])
        ]
        if item["id"] in original_questions.values()
        else "Unknown"
    )

    print(
        f"  New ID: {item['id']}, "
        f"Question: {item['question'][:30]}..."
    )

In [ ]:
import json

def update_questions_only(test_file, full_file, output_file):
    """
    Match by ID, only replace question field if different, keep other fields of test set
    
    Args:
        test_file: original test set file path
        full_file: complete corrected dataset file path  
        output_file: output file path for updated test set
    """
    
    test_data = []
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            test_data.append(data)
    
    print(f"Test set contains {len(test_data)} records")

    id_to_question = {}
    with open(full_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            id_to_question[data['id']] = data['question']
    
    print(f"Complete dataset contains {len(id_to_question)} records")
    
    # Update test set: only replace question field if different
    updated_test_data = []
    changed_count = 0
    missing_ids = []
    
    for item in test_data:
        test_id = item['id']
        if test_id in id_to_question:
            full_question = id_to_question[test_id]
            current_question = item['question']
            
            # Only update if questions are different
            if full_question != current_question:
                # Create new object, only update question field, keep other fields unchanged
                updated_item = item.copy()
                updated_item['question'] = full_question
                updated_test_data.append(updated_item)
                changed_count += 1
                print(f"Updated: ID {test_id} - Question changed")
            else:
                # Questions are same, keep original
                updated_test_data.append(item)
        else:
            missing_ids.append(test_id)
            # If ID not found, keep original data
            updated_test_data.append(item)
            print(f"Warning: ID {test_id} not found in complete dataset, keeping original question")
    
    # Save updated test set
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in updated_test_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"\nUpdate completed!")
    print(f"Successfully processed {len(updated_test_data)} records")
    print(f"Changed {changed_count} records")
    if missing_ids:
        print(f"{len(missing_ids)} IDs not found in complete dataset")
    else:
        print("All IDs successfully matched!")

# Usage example
if __name__ == "__main__":
    update_questions_only(
        test_file="/home/xy67/RFIC Chatbot/dataset/Qwen3-0.6B/Model-alignment/validation_QTSA/mcQTSA_train_change.jsonl",
        full_file='mcQTSA_new4.jsonl', 
        output_file="/home/xy67/RFIC Chatbot/dataset/Qwen3-0.6B/Model-alignment/validation_QTSA/mcQTSA_train_fixed.jsonl"
    )

In [ ]:
import json

def update_questions_only(test_file, full_file, output_file):
    """
    Match by ID, only replace question field if different, keep other fields of test set
    
    Args:
        test_file: original test set file path
        full_file: complete corrected dataset file path  
        output_file: output file path for updated test set
    """
    
    test_data = []
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            test_data.append(data)
    
    print(f"Test set contains {len(test_data)} records")

    id_to_question = {}
    with open(full_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            id_to_question[data['id']] = data['question']
    
    print(f"Complete dataset contains {len(id_to_question)} records")
    
    # Update test set: only replace question field if different
    updated_test_data = []
    changed_count = 0
    missing_ids = []
    
    for item in test_data:
        test_id = item['id']
        if test_id in id_to_question:
            full_question = id_to_question[test_id]
            current_question = item['question']
            
            # Only update if questions are different
            if full_question != current_question:
                # Create new object, only update question field, keep other fields unchanged
                updated_item = item.copy()
                updated_item['question'] = full_question
                updated_test_data.append(updated_item)
                changed_count += 1
                print(f"Updated: ID {test_id} - Question changed")
            else:
                # Questions are same, keep original
                updated_test_data.append(item)
        else:
            missing_ids.append(test_id)
            # If ID not found, keep original data
            updated_test_data.append(item)
            print(f"Warning: ID {test_id} not found in complete dataset, keeping original question")
    
    # Save updated test set
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in updated_test_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"\nUpdate completed!")
    print(f"Successfully processed {len(updated_test_data)} records")
    print(f"Changed {changed_count} records")
    if missing_ids:
        print(f"{len(missing_ids)} IDs not found in complete dataset")
    else:
        print("All IDs successfully matched!")

# Usage example
if __name__ == "__main__":
    update_questions_only(
        test_file="/home/xy67/RFIC Chatbot/dataset/Qwen3-0.6B/Model-alignment/validation_QTSA/mcQTSA_val_change.jsonl",
        full_file='mcQTSA_new4.jsonl', 
        output_file="/home/xy67/RFIC Chatbot/dataset/Qwen3-0.6B/Model-alignment/validation_QTSA/mcQTSA_val_fixed.jsonl"
    )